<a href="https://colab.research.google.com/github/SayanB58/HAAI__Cohort2_LLM_Tokenizers_Embeddings/blob/main/Programming_Assignment_3_In_context_Learning_in_LLMs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This program is build with Flan-T5-XL LLM to be able to answer a question in YES/NO using the provided context as in-context learning.

>1. The program accepts two parameters provided as a command line input.
>2. The two inputs represent the context and the question.
>3. The question output is deterministic i.e. its either YES or NO. You are required to use logits to extract the output.
>4. Output should be in upper-case: YES or NO
>5. There should be no additional output including any warning messages in the terminal.
>6. Remember that your output will be tested against test cases, therefore any deviation from the test cases will be considered incorrect during evaluation.
>7. Note that the assignment and evaluation test cases are carefully sampled from the model itself, eliminating any chance of hallucination.
>8. The context is within 30 words.
>9. Design the code to run in CPU environment
>10. It is recommended to use terminal to avoid inconsistency in output formatting.

Syntax: python template.py &lt;**CONTEXT**&gt; &lt;**QUESTION**&gt;

The following example is given for your reference:

Terminal Input: python assignment.py 'Albert has been working on his project all week. He finished the final report today and submitted it to his manager before the deadline.' 'Did Albert submit his project report on time?'
Terminal Output: YES

Terminal Input: python assignment.py 'Albert has been working on his project all week. He finished the final report today and submitted it to his manager after the deadline.' 'Did Albert submit his project report on time?'
Terminal Output: NO

Terminal Input: 'John started watered his plants every morning this week.' 'Did John water his plants yesterday morning?'
Terminal Output: YES

Terminal Input: 'John started watered his plants every morning this week.' 'Did John water his plants last month?'
Terminal Output: NO

You are expected to create some examples of your own to test the correctness of your approach.

**ALERT: No changes are allowed to import statements**

In [1]:
import sys
import torch
import transformers
from transformers import T5Tokenizer, T5ForConditionalGeneration
import re

#####
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

In [2]:
from google.colab import userdata
#userdata.get('HF_TOKEN')

**Changes allowed from here**

In [3]:
def llm_function(model,tokenizer,context,question):

    # Generate a deterministic output either 'YES' or 'NO' answering the question using the provided context

    '''
    1. Engineer the prompt using the query and the context.
    2. Tokenize the prompt.
    3. Generate output for the prompt.
    4. Extract the logit values to determine the output
    5. Format the output to be exactly YES or NO.

    Remember that there should be no additional output including any warning messages in the terminal.
    '''

    prompt = f''' Context: {context} Use the above context to answer the following question: {question} Output 1 for Yes and 2 for No: '''

    tokenized_prompt = tokenizer(prompt, return_tensors="pt").input_ids

    outputs = model.generate(
        tokenized_prompt,
        do_sample=False,
        top_p=None,
        return_dict_in_generate=True,
        output_scores=True,
        max_new_tokens=1
    )

    logit_stack = torch.stack(outputs.scores, dim=1)

    # Since our output is determinstic we must prefer logit extraction instead of generation

    A = tokenizer.encode("1", return_tensors="pt", add_special_tokens=False)[0].item()
    B = tokenizer.encode("2", return_tensors="pt", add_special_tokens=False)[0].item()

    logit_a = logit_stack[0][0][A].item()
    logit_b = logit_stack[0][0][B].item()

    logits = {'YES':logit_a, 'NO':logit_b}

    final_output = max(logits, key=logits.get)

    #print(f'Answer for Q3: {final_output}')

    return final_output

**ALERT: No changes are allowed below this comment**

In [12]:
if __name__ == '__main__':

    # context = sys.argv[1].strip().lower()
    # question = sys.argv[2].strip().lower()

    context = 'James had a dentist appointment at 3 PM on Tuesday. He arrived at the clinic at 2:45 PM.'
    question = 'Did James arrive before his appointment?'

    ##################### Loading Model and Tokenizer ########################
    tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-xl", token=userdata.get('HF_TOKEN'))
    model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-xl", token=userdata.get('HF_TOKEN'))
    ##########################################################################

    """  Call to function that will perform the computation. """
    torch.manual_seed(42)
    out = llm_function(model,tokenizer,context,question)
    print(out.strip())

    """ End to call """

YES
